# Exploratory Climate Analysis - EarthScape

EDA over the **real** documents stored in `climate_observations` (NASA POWER daily
values for monitored Pakistan + reference-city locations).

Honesty rules for this notebook:

* every number printed here comes from stored observations; if the collection is
  empty the notebook says so and stops - it never substitutes sample data;
* correlations are co-variation only, not causation;
* seasonal means describe **weather climatology** (day-to-day state averaged);
  a rising trend line is a **climate signal** and is reported separately.


In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND)); sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django; django.setup()

from common import analytics
from common.analytics import (METRICS, METRIC_LABELS, METRIC_UNITS, observation_query,
                              fetch_observations, to_frame, seasonal_baselines,
                              detect_anomalies, trend_slope, coverage_report,
                              correlation_matrix, compare_regions)
print("database mode:", analytics.__name__, "->", __import__("common.database", fromlist=["get_database"]).get_database().mode)


In [ ]:
observations = fetch_observations(observation_query())
print(f"fetched observation rows: {len(observations)}")
if not observations:
    print("NO TRAINING/ANALYSIS DATA AVAILABLE: climate_observations is empty for this scope.")
    print("Run the ingestion pipeline first (apps/ingestion) - this notebook will not fake data.")


In [ ]:
coverage = coverage_report(observations)
print("date range:", coverage["date_range"], "| locations:", coverage["locations"],
      "| regions:", coverage["regions"], "| datasets:", coverage["datasets"])
import pandas as pd
pd.DataFrame([{"metric": m, **{k: v.get(k) for k in ("populated", "missing", "pct")}}
              for m, v in coverage["completeness"].items()])


In [ ]:
frame = to_frame(observations)
metrics_present = [m for m in METRICS if m in frame.columns and frame[m].notna().any()]
if frame.empty or not metrics_present:
    print("No populated metrics in the stored record yet - distributions unavailable (honest empty result).")
else:
    dist = frame[metrics_present].describe().T
    dist["unit"] = [METRIC_UNITS[m] for m in dist.index]
    dist


In [ ]:
# Seasonality: monthly climatology of mean temperature per region (train-window-free
# descriptive EDA). seasonal_baselines computes region+month means/stds from the record.
if "temperature_c" in metrics_present:
    baselines = seasonal_baselines(observations, "temperature_c")
    rows = [{"region|month": k, "mean": v["mean"], "std": v["std"], "n": v["sample_size"]}
            for k, v in sorted(baselines.items())]
    print(f"{len(baselines)} region-month climatology groups")
    pd.DataFrame(rows).head(24)
else:
    print("temperature_c has no stored values - skipping seasonality.")


In [ ]:
# Correlations (co-variation only - explicitly NOT causation)
if not observations:
    print("Skipping correlation: no observations stored.")
else:
    corr = correlation_matrix(observations, ["temperature_c", "relative_humidity_pct",
                                             "solar_radiation_kwh_m2", "cloud_coverage_pct",
                                             "precipitation_mm"])
    print(corr.get("caveat"))
    cells = {(c["x"], c["y"]): c["coefficient"] for c in corr["cells"]}
    pd.DataFrame([[cells.get((a, b)) for b in corr["metrics"]] for a in corr["metrics"]],
                 index=corr["metrics"], columns=corr["metrics"])


In [ ]:
# Long-run climate signal, kept separate from the weather statistics above:
# monthly means of temperature per region + least-squares slope per period.
if "temperature_c" in metrics_present:
    trend = analytics.trend_series(observations, "temperature_c", frequency="monthly",
                                   group_by="region")
    points = [(p["period"], p["value"]) for p in trend["series"] if p.get("region_code") == "Punjab"]
    print("Punjab monthly periods:", len(points))
    if points:
        print(trend_slope(points))
    print("NOTE: a positive slope is a climate-trend statistic over the stored record; "
          "it does not mean any single forecast day is 'caused' by climate change.")


In [ ]:
# A first look at anomalies against the region+month climatology (2.5 sigma rule).
if metrics_present:
    result = detect_anomalies(observations, "temperature_c", method="statistical_baseline",
                              threshold_sigma=2.5)
    print("evaluated rows:", result["evaluated_observations"],
          "| anomalies:", result["anomaly_count"], "| by severity:", result["severity_counts"])
    for item in result["anomalies"][:5]:
        print("-", item["observation_date"], item["location_name"], item["value"],
              item["unit"], f"{item['deviation_score']:+.2f} sigma")
    print(result["caveat"])


## Takeaways

1. Data volume and completeness are reported exactly as stored - thin regions
   produce thin conclusions.
2. Seasonality dominates daily variance; that is why the models forecast
   anomalies relative to region-month climatology, not raw trends.
3. Correlation tables describe co-variation; causal claims are out of scope.
4. Weather vs climate: the monthly climatology and anomaly counts above are
   weather statistics; only the long-run slope speaks (weakly, station-limited)
   to climate behaviour. Retraining and evaluation: see `ml/README.md`.
